# GPT-BERT: correct causal perplexity

GPT-BERT's perplexity in `monolingual eval/{english,hindi,telugu}.py` (`score_gptbert_pll`) is a **pseudo-perplexity**: it masks one token at a time and lets the model see the *entire rest of the sequence* (both left and right context) to predict it. That's a much easier task than standard next-token prediction, so the resulting numbers (e.g. 4.8953 EN, 1.3261 HI) are not comparable to GPT-2/Sarvam/Llama's causal (left-context-only) perplexity, even though they land in the same table column.

GPT-BERT is trained with **both** a causal and a masked objective (see `gptbert multi/pretraining/dataset.py`'s `CausalDataset`, which builds a lower-triangular attention mask and shifted-by-one targets). This notebook reuses that exact convention to get a real, left-context-only causal perplexity for GPT-BERT, directly comparable to the other models.

Run this from the repo root (needs `./gpt-bert/modeling_gptbert.py` on disk, and GPU).

In [ ]:
!pip install -q transformers datasets torch tqdm

In [ ]:
import sys
import math
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import PreTrainedTokenizerFast
from tqdm.auto import tqdm

_GPTBERT_DIR = Path.cwd() / "gpt-bert"
if str(_GPTBERT_DIR) not in sys.path:
    sys.path.insert(0, str(_GPTBERT_DIR))
from modeling_gptbert import GptBertForMaskedLM

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────────
MAX_SEQ_LEN = 128
BATCH_SIZE  = 32
MAX_SAMPLES = None  # None = use full test/OS split
# ─────────────────────────────────────────────────────────────────────────

MODELS = {
    'en': 'pulipakav-1/english-gptbert_babylm2026',
    'hi': 'pulipakav-1/hindi-gptbert_babylm2026',
    'te': 'pulipakav-1/telugu-gptbert_babylm2026',
}

In [ ]:
def load_test_texts(lang):
    """Own-language held-out test set, matching monolingual eval/{lang}.py's eval_perplexity source."""
    if lang == 'en':
        ds = load_dataset('text',
                           data_files={'test': 'hf://datasets/BabyLM-community/BabyLM-Test/*.test'},
                           split='test')
        texts = [row['text'] for row in ds if row.get('text')]
    elif lang == 'hi':
        ds = load_dataset('pulipakav-1/translated-babylm-hindi', split='test', streaming=True)
        texts = [row['text'] for row in ds if row.get('text', '').strip()]
    elif lang == 'te':
        ds = load_dataset('pulipakav-1/translated-babylm-telugu', split='test', streaming=True)
        texts = [row['text'] for row in ds if row.get('text', '').strip()]
    else:
        raise ValueError(lang)
    if MAX_SAMPLES is not None and len(texts) > MAX_SAMPLES:
        import random; random.seed(42)
        texts = random.sample(texts, MAX_SAMPLES)
    print(f'  [{lang}] Test set: {len(texts):,} lines')
    return texts


def load_os_texts(lang1, lang2, side):
    """Load OPUS-100 parallel text for one language side (same source as eval_opensubtitles.ipynb)."""
    config = f"{lang1}-{lang2}"
    print(f'Loading Helsinki-NLP/opus-100 ({config}), extracting {side} side ...')
    ds = load_dataset('Helsinki-NLP/opus-100', config, split='train', streaming=True)
    texts = []
    for row in ds:
        text = row['translation'][side].strip()
        if text:
            texts.append(text)
        if MAX_SAMPLES is not None and len(texts) >= MAX_SAMPLES:
            break
    print(f'  Loaded {len(texts):,} lines')
    return texts

## Causal perplexity (correct, comparable metric)

Reuses GPT-BERT's own causal-mode convention from `gptbert multi/pretraining/dataset.py::CausalDataset`:
- `input_ids = [CLS] + tokens` (seq-first tensor `[L, B]`, matching the model's internal layout — see `score_gptbert_pll` in `monolingual eval/*.py`, which uses the same seq-first convention by calling `model.embedding` / `model.transformer` / `model.classifier` directly rather than going through the HF `forward()` wrapper, which is never exercised elsewhere in this codebase).
- `labels = tokens + [-100]` (shifted by one: position *i*'s target is the token that comes right after input position *i*).
- attention mask: lower-triangular (each position may only attend to itself and earlier positions) plus padding, batch-first `[B, 1, L, L]`, `True` = blocked (matches `MaskedSoftmax`'s convention).

In [ ]:
def compute_causal_perplexity(model, tokenizer, texts, cls_id, pad_id, max_seq_len=MAX_SEQ_LEN, batch_size=BATCH_SIZE):
    model.eval()
    total_nll, total_tokens = 0.0, 0

    for i in tqdm(range(0, len(texts), batch_size), leave=False):
        batch_texts = texts[i:i + batch_size]
        token_lists = [tokenizer.encode(t, add_special_tokens=False)[:max_seq_len - 1] for t in batch_texts]
        token_lists = [t for t in token_lists if len(t) > 0]
        if not token_lists:
            continue
        B = len(token_lists)
        L = max(len(t) for t in token_lists) + 1  # +1 for CLS

        input_ids = torch.full((L, B), pad_id, dtype=torch.long)
        labels    = torch.full((L, B), -100,   dtype=torch.long)
        valid_lens = torch.zeros(B, dtype=torch.long)

        for b, tok in enumerate(token_lists):
            n = len(tok)
            seq = torch.tensor([cls_id] + tok, dtype=torch.long)   # length n+1
            tgt = torch.tensor(tok + [-100],   dtype=torch.long)   # length n+1
            input_ids[:n + 1, b] = seq
            labels[:n + 1, b]    = tgt
            valid_lens[b] = n + 1

        input_ids = input_ids.to(DEVICE)
        labels    = labels.to(DEVICE)
        valid_lens = valid_lens.to(DEVICE)

        causal_block = torch.triu(torch.ones(L, L, dtype=torch.bool, device=DEVICE), diagonal=1)  # True where key > query
        pad_block = torch.arange(L, device=DEVICE).unsqueeze(0) >= valid_lens.unsqueeze(1)          # [B, L] True = padding column
        pad_block = pad_block.unsqueeze(1).expand(-1, L, -1)                                        # [B, L(query), L(key)]
        mask_out = (causal_block.unsqueeze(0) | pad_block).unsqueeze(1)                              # [B, 1, L, L]

        with torch.no_grad():
            static_emb, rel_emb = model.embedding(input_ids)
            hidden = model.transformer(static_emb, mask_out, rel_emb)
            logits = model.classifier(hidden, labels)

        gold = labels.flatten()
        gold = gold[gold != -100]
        if gold.numel() == 0:
            continue
        nll_sum = F.cross_entropy(logits, gold, reduction='sum').item()
        total_nll    += nll_sum
        total_tokens += gold.numel()

    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')

## Pseudo-perplexity (old metric, kept for side-by-side comparison)

Identical to `score_gptbert_pll` in `monolingual eval/*.py` — one text at a time, one masked position at a time, full bidirectional context.

In [ ]:
def score_gptbert_pll(model, token_ids, cls_id, mask_id):
    full_ids = [cls_id] + token_ids
    seq_len  = len(full_ids)
    n        = seq_len - 1
    if n == 0:
        return float('-inf')
    base     = torch.tensor(full_ids, dtype=torch.long, device=DEVICE)
    input_t  = base.unsqueeze(1).expand(-1, n).clone()
    labels_t = torch.full((seq_len, n), -100, dtype=torch.long, device=DEVICE)
    for j in range(n):
        pos = j + 1
        input_t[pos, j]  = mask_id
        labels_t[pos, j] = base[pos]
    attn = torch.zeros(n, 1, seq_len, seq_len, dtype=torch.bool, device=DEVICE)
    with torch.no_grad():
        static_emb, rel_emb = model.embedding(input_t)
        hidden               = model.transformer(static_emb, attn, rel_emb)
        logits               = model.classifier(hidden, labels_t)
    log_probs = F.log_softmax(logits, dim=-1)
    gold      = base[1:].to(DEVICE)
    return log_probs[torch.arange(n, device=DEVICE), gold].sum().item()


def compute_pll_perplexity(model, tokenizer, texts, cls_id, mask_id, max_seq_len=MAX_SEQ_LEN):
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for text in tqdm(texts, leave=False):
        token_ids = tokenizer.encode(text, add_special_tokens=False)[:max_seq_len]
        if not token_ids:
            continue
        pll = score_gptbert_pll(model, token_ids, cls_id, mask_id)
        total_nll    += -pll
        total_tokens += len(token_ids)
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')

In [ ]:
# Load OS data once (English side from EN-HI; Hindi from EN-HI; Telugu from EN-TE)
os_data = {}
os_data['en'] = load_os_texts('en', 'hi', 'en')
os_data['hi'] = load_os_texts('en', 'hi', 'hi')
os_data['te'] = load_os_texts('en', 'te', 'te')

In [ ]:
results = {}

for lang, repo in MODELS.items():
    print(f'\n=== {lang} ({repo}) ===')
    tokenizer = PreTrainedTokenizerFast.from_pretrained(repo)
    cls_id  = tokenizer.convert_tokens_to_ids('<s>')
    mask_id = tokenizer.convert_tokens_to_ids('<mask>')
    pad_id  = tokenizer.convert_tokens_to_ids('<pad>')
    if pad_id is None or pad_id < 0:
        pad_id = 0
    model = GptBertForMaskedLM.from_pretrained(repo).eval().to(DEVICE)

    test_texts = load_test_texts(lang)

    causal_test = compute_causal_perplexity(model, tokenizer, test_texts, cls_id, pad_id)
    causal_os   = compute_causal_perplexity(model, tokenizer, os_data[lang], cls_id, pad_id)
    pll_test    = compute_pll_perplexity(model, tokenizer, test_texts, cls_id, mask_id)

    results[lang] = {
        'causal_test': round(causal_test, 4),
        'causal_os':   round(causal_os, 4),
        'pll_test':    round(pll_test, 4),
    }
    print(f"  Causal Test:  {causal_test:.4f}")
    print(f"  Causal OS:    {causal_os:.4f}")
    print(f"  PLL Test (old, not comparable): {pll_test:.4f}")

    del model
    torch.cuda.empty_cache()

In [ ]:
print('\n=== GPT-BERT Perplexity Results ===')
print(f'{"Lang":<6} {"Causal-Test":>12} {"Causal-OS":>12} {"PLL-Test (old)":>16}')
print('-' * 50)
for lang in MODELS:
    r = results[lang]
    print(f"{lang:<6} {r['causal_test']:>12} {r['causal_os']:>12} {r['pll_test']:>16}")